# Bronze Layer — retention (keep 7 days, only reads already in Silver)

Added 2026-11-02 (datle): Silver now keeps **every read** (grain id × `snapshot_ts`), so Bronze is only a short buffer to rerun Silver after a failure. Runs at the end of the daily job, after every Silver task that reads Bronze.

**Rule per Bronze table:** delete rows with `ingest_date` older than `RETENTION_DAYS` (7) **and** whose timestamp is at or below the newest read of its Silver table (so a Silver task that has been failing never loses data: its rows stay until Silver has them).

| Bronze | Silver that holds it | Timestamp compared |
| --- | --- | --- |
| `videos_raw`, `channels_raw`, `comments_raw`, `places_raw`, `search_keywords_raw` | `stg_videos`, `stg_channels`, `stg_comments`, `stg_places`, `stg_search_keywords` | `COALESCE(updated_at, created_at)` vs `snapshot_ts` |
| `place_search_results_raw` | `stg_place_search_results` | `fetched_at` vs `snapshot_ts` |
| `search_results_raw` | `stg_search_results` | `fetched_at` |
| `search_runs_raw` | none (Lakebase `search_runs` is append-only and keeps every run) | age only |
| `osm_pois_raw` | `stg_osm_restaurants` (latest snapshot) | keep the last `OSM_KEEP_SNAPSHOTS` (4) weekly snapshots, older than 7 days deleted |

In [ ]:
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

RETENTION_DAYS = 7
OSM_KEEP_SNAPSHOTS = 4
UPSERTED = "COALESCE(updated_at, created_at)"
# bronze table -> (silver table, bronze timestamp expression, silver timestamp column); None = age only
RULES = {
    "videos_raw": ("stg_videos", UPSERTED, "snapshot_ts"),
    "channels_raw": ("stg_channels", UPSERTED, "snapshot_ts"),
    "comments_raw": ("stg_comments", UPSERTED, "snapshot_ts"),
    "places_raw": ("stg_places", UPSERTED, "snapshot_ts"),
    "search_keywords_raw": ("stg_search_keywords", UPSERTED, "snapshot_ts"),
    "place_search_results_raw": ("stg_place_search_results", "fetched_at", "snapshot_ts"),
    "search_results_raw": ("stg_search_results", "fetched_at", "fetched_at"),
    "search_runs_raw": None,
}
print(f"Bronze {BRONZE}: keep {RETENTION_DAYS} days, only reads already in Silver {SILVER}")

In [ ]:
cutoff = f"date_sub(to_date(from_utc_timestamp(current_timestamp(), 'Asia/Ho_Chi_Minh')), {RETENTION_DAYS})"
for bronze, rule in RULES.items():
    table = f"{BRONZE}.{bronze}"
    if not spark.catalog.tableExists(table):
        continue
    before = spark.table(table).count()
    if rule is None:
        spark.sql(f"DELETE FROM {table} WHERE ingest_date < {cutoff}")
    else:
        silver, ts_expr, silver_ts = rule
        silver_table = f"{SILVER}.{silver}"
        if not spark.catalog.tableExists(silver_table) or silver_ts not in spark.table(silver_table).columns:
            print(f"⏭️ {table}: {silver_table} not ready (no {silver_ts}), nothing deleted")
            continue
        wm = spark.sql(f"SELECT MAX({silver_ts}) AS wm FROM {silver_table}").collect()[0]["wm"]
        if wm is None:
            print(f"⏭️ {table}: {silver_table} is empty, nothing deleted")
            continue
        spark.sql(f"""DELETE FROM {table}
                      WHERE ingest_date < {cutoff} AND {ts_expr} <= '{wm.strftime("%Y-%m-%d %H:%M:%S.%f")}'""")
    after = spark.table(table).count()
    print(f"🧹 {table}: {before} → {after} rows (deleted {before - after})")

osm = f"{BRONZE}.osm_pois_raw"
if spark.catalog.tableExists(osm):
    before = spark.table(osm).count()
    spark.sql(f"""
        DELETE FROM {osm}
        WHERE ingest_date < {cutoff}
          AND ingest_ts NOT IN (SELECT ingest_ts FROM (SELECT DISTINCT ingest_ts FROM {osm}) ORDER BY ingest_ts DESC LIMIT {OSM_KEEP_SNAPSHOTS})
    """)
    print(f"🧹 {osm}: {before} → {spark.table(osm).count()} rows (keeps the last {OSM_KEEP_SNAPSHOTS} snapshots)")
print("\n--- Bronze retention complete ---")